In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Export Floor Plan to DXF (Rhino)

Reads a floor plan from the MSD dataset and exports room polygons as a DXF file.  
Each room type gets its own layer. Open the output in Rhino directly.

**Requires:** `ezdxf` — install with `pip install ezdxf`

In [2]:
import pickle
import ezdxf
from ezdxf.enums import TextEntityAlignment

DATA_PATH   = "../01_dataset/data/modified-swiss-dwellings-v2/train/"
OUTPUT_PATH = "../01_dataset/floor_plan/"   # saves next to this notebook in 01_dataset/floor_plan/

ROOM_NAMES = {
    0: 'Balcony', 1: 'Bathroom', 2: 'Bedroom', 3: 'Corridor',
    4: 'Dining',  5: 'Kitchen',  6: 'LivingRoom', 7: 'Storeroom', 8: 'Other'
}

# AutoCAD color index (ACI) per room type
ROOM_ACI = {
    0: 140,  # Balcony     — light blue
    1: 92,   # Bathroom    — green
    2: 50,   # Bedroom     — yellow
    3: 30,   # Corridor    — orange
    4: 210,  # Dining      — purple
    5: 10,   # Kitchen     — red
    6: 150,  # Living Room — cyan
    7: 8,    # Storeroom   — grey
    8: 255,  # Other       — white
}

def load(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def get_coords(geom):
    if geom is None:
        return None
    return list(geom.exterior.coords) if hasattr(geom, 'exterior') else list(geom)

print('Ready. Set PLAN_ID in the next cell.')

Ready. Set PLAN_ID in the next cell.


In [3]:
# ── Configure ─────────────────────────────────────────────────────────────────
PLAN_ID = '10000'   # change to any plan ID from the dataset

g_out = load(DATA_PATH + f'graph_out/{PLAN_ID}.pickle')
g_in  = load(DATA_PATH + f'graph_in/{PLAN_ID}.pickle')

print(f'Plan {PLAN_ID} — {g_out.number_of_nodes()} rooms, {g_in.number_of_edges()} connections')

from collections import Counter
counts = Counter(d['room_type'] for _, d in g_out.nodes(data=True))
for rt, n in sorted(counts.items()):
    print(f'  {ROOM_NAMES[rt]:12s} x{n}')

Plan 10000 — 41 rooms, 39 connections
  Balcony      x13
  Bedroom      x5
  Dining       x7
  Kitchen      x2
  LivingRoom   x2
  Storeroom    x6
  Other        x6


In [4]:
# ── Export to DXF ─────────────────────────────────────────────────────────────
doc = ezdxf.new(dxfversion='R2010')
doc.header['$INSUNITS'] = 6   # metres
msp = doc.modelspace()

# Create one layer per room type
for rt, name in ROOM_NAMES.items():
    doc.layers.add(name=name, color=ROOM_ACI[rt])
doc.layers.add(name='Labels',               color=7)    # white
doc.layers.add(name='Graph_nodes',          color=5)    # blue
doc.layers.add(name='Graph_edges_door',     color=150)  # cyan
doc.layers.add(name='Graph_edges_entrance', color=10)   # red

# ── Room polygons + labels ──
pos = {}  # node → (cx, cy) for graph overlay

for node, data in g_out.nodes(data=True):
    coords = get_coords(data.get('geometry'))
    if coords is None:
        continue
    rt   = data.get('room_type', 8)
    name = ROOM_NAMES[rt]

    pts2d = [(x, y) for x, y in coords]
    msp.add_lwpolyline(pts2d, close=True, dxfattribs={'layer': name})

    c = data.get('centroid')
    if c is not None:
        cx = c.x if hasattr(c, 'x') else c[0]
        cy = c.y if hasattr(c, 'y') else c[1]
        pos[node] = (cx, cy)
        t = msp.add_text(name, dxfattribs={'layer': 'Labels', 'height': 0.15})
        t.set_placement((cx, cy), align=TextEntityAlignment.MIDDLE_CENTER)

# ── Graph overlay (toggle layer visibility in Rhino to show/hide) ──
for node, (cx, cy) in pos.items():
    msp.add_circle((cx, cy), radius=0.08, dxfattribs={'layer': 'Graph_nodes'})

for u, v, edata in g_in.edges(data=True):
    if u in pos and v in pos:
        layer = 'Graph_edges_entrance' if edata.get('connectivity') == 'entrance' else 'Graph_edges_door'
        msp.add_line(pos[u], pos[v], dxfattribs={'layer': layer})

# ── Save ──
out_file = OUTPUT_PATH + f'plan_{PLAN_ID}.dxf'
doc.saveas(out_file)
print(f'Saved → {out_file}')
print('Open in Rhino: File > Open, or drag and drop.')
print('Layers: one per room type + Labels + Graph_nodes + Graph_edges_door/entrance')

Saved → ./plan_10000.dxf
Open in Rhino: File > Open, or drag and drop.
Layers: one per room type + Labels + Graph_nodes + Graph_edges_door/entrance
